In [1]:
import os
import tarfile
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import download_asset
from torch.utils.data import Dataset, DataLoader
from jiwer import wer
from tqdm.auto import tqdm
import whisper

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 10
    lr = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tar_path = "ru_train_0_19.tar"
    tsv_path = "train(1).tsv"
    extract_dir = "./ru_train_data"

    n_fft = 512
    hop_length = 160
    win_length = 400
    num_freqs = n_fft // 2 + 1
    T_frames = 192
    target_samples = 16384*2

In [4]:
class GlobalLayerNorm(nn.Module):
    def __init__(self, channels, eps=1e-8):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        var = x.var(dim=(1, 2, 3), unbiased=False, keepdim=True)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta

class FTB_Lite(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.freq_conv = nn.Conv2d(in_channels, in_channels, kernel_size=(1, 3), padding=(0, 1))
        self.bn = nn.BatchNorm2d(in_channels)

    def forward(self, x):
        return F.relu(self.bn(self.freq_conv(x)))

class TSB_Lite(nn.Module):
    def __init__(self, C_A=32, C_P=16):
        super().__init__()
        self.ftb = FTB_Lite(C_A)
        self.convA = nn.Sequential(
            nn.Conv2d(C_A, C_A, kernel_size=3, padding=1),
            nn.BatchNorm2d(C_A),
            nn.ReLU(),
            nn.Conv2d(C_A, C_A, kernel_size=(7, 1), padding=(3, 0)), # Ядро 25 -> 7
            nn.BatchNorm2d(C_A),
            nn.ReLU()
        )
        self.convP = nn.Sequential(
            GlobalLayerNorm(C_P),
            nn.Conv2d(C_P, C_P, kernel_size=(7, 1), padding=(3, 0)) # Ядро 25 -> 7
        )
        self.gate_P_to_A = nn.Conv2d(C_P, C_A, kernel_size=1)
        self.gate_A_to_P = nn.Conv2d(C_A, C_P, kernel_size=1)

    def forward(self, sa, sp):
        sa_out = self.convA(self.ftb(sa))
        sp_out = self.convP(sp)
        sa_next = sa_out * torch.tanh(self.gate_P_to_A(sp_out))
        sp_next = sp_out * torch.tanh(self.gate_A_to_P(sa_out))
        return sa_next, sp_next

class PHASEN(nn.Module):
    def __init__(self):
        super().__init__()
        self.F = 257
        C_A, C_P = 32, 16

        self.init_A = nn.Sequential(
            nn.Conv2d(2, C_A, kernel_size=3, padding=1),
            nn.BatchNorm2d(C_A), nn.ReLU()
        )
        
        self.init_P = nn.Sequential(
            nn.Conv2d(2, C_P, kernel_size=3, padding=1)
        )

        self.tsb = TSB_Lite(C_A, C_P)

        self.out_A_conv = nn.Conv2d(C_A, 4, kernel_size=1)

        self.rnn = nn.GRU(
            input_size=self.F * 4, 
            hidden_size=128, 
            bidirectional=True, 
            batch_first=True
        )
        
        self.fc = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, self.F),
            nn.Sigmoid()
        )

        self.out_P_conv = nn.Conv2d(C_P, 2, kernel_size=1)

    def forward(self, x):
        B, C, T, F_dim = x.shape

        sa = self.init_A(x)
        sp = self.init_P(x)

        sa, sp = self.tsb(sa, sp)

        mask = self.out_A_conv(sa).permute(0, 2, 1, 3).reshape(B, T, -1)
        mask, _ = self.rnn(mask)
        mask = self.fc(mask).unsqueeze(1)

        phase_raw = self.out_P_conv(sp)
        phase = phase_raw / (torch.norm(phase_raw, dim=1, keepdim=True) + 1e-8)

        mag_in = torch.norm(x, dim=1, keepdim=True)

        return (mag_in * mask) * phase


class PHASENLoss(nn.Module):
    def __init__(self, power=0.3, eps=1e-8):
        super().__init__()
        self.power = power
        self.eps = eps

    def forward(self, pred, target):
        mag_pred = torch.sqrt(pred[:, 0:1, :, :]**2 + pred[:, 1:2, :, :]**2 + self.eps)
        mag_target = torch.sqrt(target[:, 0:1, :, :]**2 + target[:, 1:2, :, :]**2 + self.eps)

        mag_pred_cprs = mag_pred ** self.power
        mag_target_cprs = mag_target ** self.power
        L_a = F.mse_loss(mag_pred_cprs, mag_target_cprs)

        pred_cprs_real = mag_pred_cprs * pred[:, 0:1, :, :] / mag_pred
        pred_cprs_imag = mag_pred_cprs * pred[:, 1:2, :, :] / mag_pred
        pred_cprs = torch.cat([pred_cprs_real, pred_cprs_imag], dim=1)

        target_cprs_real = mag_target_cprs * target[:, 0:1, :, :] / mag_target
        target_cprs_imag = mag_target_cprs * target[:, 1:2, :, :] / mag_target
        target_cprs = torch.cat([target_cprs_real, target_cprs_imag], dim=1)

        L_p = F.mse_loss(pred_cprs, target_cprs)

        loss = 0.5 * L_a + 0.5 * L_p
        return loss

In [5]:
def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_p = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noi_p = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_p = sig_p / (10 ** (snr_db / 10))
    return torch.sqrt(target_p / (noi_p + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[..., :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [6]:
class WaveformDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True, max_len_sec=3.0):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        self.max_len_sec = max_len_sec

        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav, sr = torchaudio.load(file_path)

        if sr != Config.sr:
            wav = T.Resample(sr, Config.sr)(wav)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        max_samples = int(self.max_len_sec * Config.sr)

        if self.is_train:
            if wav.shape[-1] > max_samples:
                start = random.randint(0, wav.shape[-1] - max_samples)
                wav = wav[:, start:start + max_samples]
            else:
                wav = F.pad(wav, (0, max_samples - wav.shape[-1]))
        else:
            if wav.shape[-1] > max_samples:
                wav = wav[:, :max_samples]
            else:
                wav = F.pad(wav, (0, max_samples - wav.shape[-1]))

        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean

        return noisy, clean, self.ref_dict[os.path.basename(file_path)]

In [7]:
if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar:
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM
BABBLE_WAVEFORM, sr_b = torchaudio.load(babble_path)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM
RIR_WAVEFORM, sr_r = torchaudio.load(rir_path)
RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

/tmp/ipykernel_54353/1144183596.py:9: UserWarning: torchaudio.utils.download.download_asset has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  babble_path = download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packa

In [8]:
dataset = WaveformDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size],generator = generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)

In [9]:
model = PHASEN().to(Config.device)

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=Config.lr)

In [11]:
criterion = PHASENLoss().to(Config.device)

In [12]:
window = torch.hann_window(Config.win_length).to(Config.device)

In [94]:
for epoch in range(1, Config.epochs + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for noisy, clean, _ in pbar:
        noisy = noisy.squeeze(1).to(Config.device)
        clean = clean.squeeze(1).to(Config.device)

        X = torch.stft(noisy, Config.n_fft, Config.hop_length, Config.win_length, window, return_complex=True)
        Y = torch.stft(clean, Config.n_fft, Config.hop_length, Config.win_length, window, return_complex=True)

        X_in = torch.stack([X.real, X.imag], dim=1).transpose(2, 3)
        Y_target = torch.stack([Y.real, Y.imag], dim=1).transpose(2, 3)

        optimizer.zero_grad()
        pred = model(X_in)

        loss = criterion(pred, Y_target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    if epoch % 5 == 0:
        torch.save(model.state_dict(), f"phasen_epoch_{epoch}.pth")

Epoch 1:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 6:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/5954 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/5954 [00:00<?, ?it/s]

In [13]:
model.load_state_dict(torch.load('phasen_epoch_50.pth'))

seed_everything(42)

In [ ]:
def evaluate_model(model, device, val_dataset, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}
    window = torch.hann_window(Config.win_length).to(device)

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):
            _, clean_wav, ref_text = val_dataset[idx]
            ref_text = clean_text(ref_text)

            for n_type in noise_types:
                noisy_wav = apply_noise(clean_wav, force_type=n_type, file_seed=idx).to(device)

                X = torch.stft(noisy_wav, Config.n_fft, Config.hop_length, Config.win_length, window, return_complex=True)

                X_in = torch.stack([X.real, X.imag], dim=1).transpose(2, 3)
                
                pred = model(X_in)

                E_real = pred[:, 0, :, :].transpose(1, 2)
                E_imag = pred[:, 1, :, :].transpose(1, 2)
                E_complex = torch.complex(E_real, E_imag)

                denoised_wav = torch.istft(E_complex, Config.n_fft, Config.hop_length, Config.win_length, window, length=noisy_wav.shape[-1])
                denoised_wav = torch.clamp(denoised_wav, min=-1.0, max=1.0)

                noisy_np = noisy_wav.squeeze().cpu().numpy()
                denoised_np = denoised_wav.squeeze().cpu().numpy().astype(np.float32)

                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']

                stats[n_type]["wer_n"].append(wer(ref_text, clean_text(t_n)))
                stats[n_type]["wer_d"].append(wer(ref_text, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]["wer_n"]), np.mean(stats[n]["wer_d"])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate_model(model, Config.device, val_ds, limit=20)

In [14]:
from jiwer import process_words

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    window = torch.hann_window(Config.win_length).to(device)

    noise_types = ['babble', 'rir', 'white']

    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}

    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0)

            for n_type in noise_types:
                noisy_tensor_cpu = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx)
                noisy_tensor = noisy_tensor_cpu.to(device)

                X = torch.stft(
                    noisy_tensor.squeeze(1), 
                    Config.n_fft, 
                    Config.hop_length, 
                    Config.win_length, 
                    window, 
                    return_complex=True
                )
                
                X_in = torch.stack([X.real, X.imag], dim=1).transpose(2, 3)

                S_hat = model(X_in)

                E_real = S_hat[:, 0, :, :].transpose(1, 2)
                E_imag = S_hat[:, 1, :, :].transpose(1, 2)
                E_complex = torch.complex(E_real, E_imag)

                denoised_tensor = torch.istft(
                    E_complex, 
                    Config.n_fft, 
                    Config.hop_length, 
                    Config.win_length, 
                    window, 
                    length=noisy_tensor.shape[-1]
                )
                denoised_tensor = torch.clamp(denoised_tensor, min=-1.0, max=1.0)

                noisy_np = noisy_tensor.squeeze().cpu().numpy()
                denoised_np = denoised_tensor.squeeze().cpu().numpy().astype(np.float32)

                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']

                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)

                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits

                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    for key in ["wer_n", "s_n", "d_n", "i_n"]:
                        stats[n_type][key].append(0.0)

                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits

                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    for key in ["wer_d", "s_d", "d_d", "i_d"]:
                        stats[n_type][key].append(0.0)

    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 65)

    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])

        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])

        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"

        print(f"{n_type:<8} | {str_noisy:<25} | {str_denois:<25} | {wer_n - wer_d:<8.4f}")

seed_everything(42)
evaluate_and_listen_components(model, Config.device, val_ds, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
-----------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.4448 (0.0672/0.3730/0.0045) | 0.0928  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0266 (0.3892/0.6052/0.0321) | -0.0266 
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.6209 (0.2344/0.3736/0.0129) | -0.0539 
